# July 2024 ST corridor: standard 4/3/2 vs single-point heads Vecchia pseudo-likelihood

Reference notebook: `real_july2024_st_corridor_spectral_norm_2days_matern_gc_nugget0_061626.ipynb`.

This notebook fits exactly one July 2024 day with generalized Cauchy `alpha=0.75, beta=1`, nugget fixed 0, then compares:

- `standard_432`: lightweight corridor-width 4x4 lag 4/3/2 block Vecchia.
- `heads200_points_x8_plus_432`: take the first 200 max-min ordered spatial centers, use **one single center-nearest data point per center per time slot** (not the whole 4x4 block), build one exact full-GP likelihood on those up-to-1600 points, then add that raw likelihood contribution to the lightweight 4/3/2 Vecchia objective.

Important: the heads version is intentionally a pseudo-likelihood diagnostic, not the exact likelihood for a strict ordering. The head points are **not removed from the original Vecchia tail**; the original Vecchia objective is kept as if the head likelihood did not exist, and then the exact head likelihood is added. The reported loss is normalized by the augmented pseudo-observation count and is useful as a within-notebook diagnostic, but should not be treated as a theorem-level likelihood comparison.


In [1]:
from __future__ import annotations

import contextlib
import gc
import io
import json
import math
import sys
import time
import types
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ST_SCRIPT = ROOT / 'Exercises/st_model/day/amarel_simulation/space_time/spectral_analysis/real_july_st_corridor_spectral_profile_v2_061426.py'
SCRIPT_DIR = ST_SCRIPT.parent
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import real_july_st_corridor_spectral_profile_v2_061426 as st
from GEMS_TCO.vecchia_st_generalized_cauchy import RealDataCorridorWidth4x4Lag432NoNuggetGeneralizedCauchyFit

pd.set_option('display.max_columns', 120)
print('Loaded ST reference:', ST_SCRIPT)

Loaded ST reference: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/amarel_simulation/space_time/spectral_analysis/real_july_st_corridor_spectral_profile_v2_061426.py


In [2]:
# Main knobs. DAY_IDX=0 means 2024-07-01. Change this single value if you want another exact day.
YEAR = 2024
MONTH = 7
DAY_IDX = 0
MODEL_VARIANT = 'gc_a075_b1'
HEADS_PER_TIME = 200
RANDOM_SEED = 20260616

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626')
OUT_DIR.mkdir(parents=True, exist_ok=True)

args = st.build_arg_parser().parse_args([])
args.real_years = [str(YEAR)]
args.month = MONTH
args.days = str(DAY_IDX)
args.model_variants = [MODEL_VARIANT]
args.out_dir = OUT_DIR
args.monthly_out_dir = OUT_DIR / 'monthly_plots_top'
args.suppress_fit_prints = True
args.n_bins = 80
args.skip_existing = False
args.device = None
args.require_cuda = False
args.cuda_fallback = 'cpu'
args.target_chunk_size = 128
args.lbfgs_steps = 8
args.lbfgs_eval = 20
args.lbfgs_history = 10
args.grad_tol = 1e-5

# Force exactly one day without editing the reference script.
def _exact_one_day_parser(_text):
    return [DAY_IDX]

st.parse_day_idxs = _exact_one_day_parser
st.load_real_assets.__globals__['parse_day_idxs'] = _exact_one_day_parser

device = st.resolve_device(args)
print('Device:', device)
print('Selected day_idx/date:', DAY_IDX, f'{YEAR}-{MONTH:02d}-{DAY_IDX + 1:02d}')
print('Model:', MODEL_VARIANT, st.variant_spec(MODEL_VARIANT))
print('Output:', OUT_DIR)

Device: cpu
Selected day_idx/date: 0 2024-07-01
Model: gc_a075_b1 {'family': 'cauchy', 'smooth': nan, 'gc_alpha': 0.75, 'gc_beta': 1.0, 'label': 'GC a=0.75 b=1 nugget0', 'model_variant': 'gc_a075_b1'}
Output: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626


In [3]:
assets = st.load_real_assets(args)
assert len(assets) == 1, [(a['day'], a['day_idx']) for a in assets]
asset = assets[0]
print('Loaded asset:', asset['day'], 'grid=', asset['grid_coords_np'].shape, 'time slots=', len(asset['source_map']))


Loading real July data for 2024-07
--- Global Monthly Mean for 2024-7: 257.9726 ---
n hourly slots: 248 grid: (18126, 2) monthly_mean: 257.9726104252314
Loaded asset: 2024-07-01 grid= (18126, 2) time slots= 8


In [4]:
def _sync_if_cuda(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)


def select_head_center_rows(model, heads_per_time: int = 200):
    """Pick one single center-nearest observed point from each of the first max-min ordered centers for each time slot."""
    all_data = [v.to(model.device, dtype=torch.float32) for v in model.input_map.values()]
    n_grid = int(all_data[0].shape[0])
    real_data = torch.cat(all_data, dim=0).contiguous()
    if not model.cluster_points:
        raise RuntimeError('Run model.precompute_conditioning_sets() before selecting heads.')
    centroids = np.asarray(model.cluster_centroids, dtype=np.float64)
    head_local = []
    for cluster_idx in range(min(int(heads_per_time), len(model.cluster_points))):
        pts = np.asarray(model.cluster_points[cluster_idx], dtype=np.int64)
        center = centroids[cluster_idx]
        grid_xy = np.asarray(model.grid_coords[:n_grid], dtype=np.float64)[pts]
        j = int(np.argmin(np.sum((grid_xy - center) ** 2, axis=1)))
        head_local.append(int(pts[j]))
    head_idx = []
    valid_mask = ~torch.isnan(real_data[:, 2])
    for time_idx in range(len(all_data)):
        offset = time_idx * n_grid
        for local_idx in head_local:
            global_idx = offset + int(local_idx)
            if bool(valid_mask[global_idx].detach().cpu().item()):
                head_idx.append(global_idx)
    head_idx_t = torch.tensor(head_idx, device=model.device, dtype=torch.long)
    return real_data[head_idx_t].contiguous().to(torch.float64), head_idx_t


def install_head_augmented_pseudolikelihood(model, heads_per_time: int = 200):
    """Add exact single-point head covariance terms to the existing cluster-tail GLS/NLL accumulation; do not remove those points from the original Vecchia tail."""
    heads_data, head_idx = select_head_center_rows(model, heads_per_time=heads_per_time)
    model.Heads_data = heads_data
    model._heads_tensor_stored = head_idx
    model.n_heads_augmented = int(heads_data.shape[0])
    model.n_heads_per_time_requested = int(heads_per_time)
    model._heads_accumulate_calls = 0
    model._heads_accumulate_s = 0.0
    tail_accumulate = model._accumulate_gls_stats
    tail_cluster_summary = model.cluster_summary

    def _accumulate_with_heads(self, params, include_y_quad=True, catch_cholesky=False):
        tail_stats = tail_accumulate(params, include_y_quad=include_y_quad, catch_cholesky=catch_cholesky)
        if tail_stats is None:
            return None
        XT_Sinv_X, XT_Sinv_y, yT_Sinv_y, log_det, total_N = tail_stats
        if self.Heads_data is None or self.Heads_data.shape[0] == 0:
            return XT_Sinv_X, XT_Sinv_y, yT_Sinv_y, log_det, total_N
        _sync_if_cuda(self.device)
        heads_t0 = time.time()
        X_h, y_h = self._head_design_response()
        cov = self.matern_cov_aniso_STABLE_log_reparam(params, self.Heads_data, self.Heads_data)
        try:
            L = torch.linalg.cholesky(cov)
        except torch.linalg.LinAlgError:
            if catch_cholesky:
                self._log_cholesky_failure(params, 'Heads exact block')
                return None
            raise
        Z_X = torch.linalg.solve_triangular(L, X_h, upper=False)
        Z_y = torch.linalg.solve_triangular(L, y_h, upper=False)
        log_det = log_det + 2.0 * torch.sum(torch.log(torch.diagonal(L)))
        XT_Sinv_X = XT_Sinv_X + Z_X.T @ Z_X
        XT_Sinv_y = XT_Sinv_y + Z_X.T @ Z_y
        if include_y_quad:
            yT_Sinv_y = yT_Sinv_y + (Z_y.T @ Z_y).squeeze()
        total_N = int(total_N) + int(self.Heads_data.shape[0])
        _sync_if_cuda(self.device)
        self._heads_accumulate_calls += 1
        self._heads_accumulate_s += time.time() - heads_t0
        return XT_Sinv_X, XT_Sinv_y, yT_Sinv_y, log_det, total_N

    def _cluster_summary_with_heads(self):
        out = dict(tail_cluster_summary())
        out['heads_per_time_requested'] = int(self.n_heads_per_time_requested)
        out['n_heads_exact'] = int(self.n_heads_augmented)
        out['n_pseudo_observations'] = int(out.get('n_target_points', 0)) + int(self.n_heads_augmented)
        out['heads_accumulate_calls'] = int(getattr(self, '_heads_accumulate_calls', 0))
        out['heads_accumulate_s'] = float(getattr(self, '_heads_accumulate_s', 0.0))
        out['heads_strategy'] = 'center_points_first_ordered_clusters'
        return out

    model._accumulate_gls_stats = types.MethodType(_accumulate_with_heads, model)
    model.cluster_summary = types.MethodType(_cluster_summary_with_heads, model)
    return model

In [5]:
def fit_asset_strategy(asset, spec, strategy_name: str, fit_id: int):
    source_map = {
        k: v.to(device=device, dtype=st.DTYPE, non_blocking=True).contiguous()
        for k, v in asset['source_map'].items()
    }
    grid_coords_np = np.asarray(asset['grid_coords_np'], dtype=np.float64)
    n_valid, n_total, valid_by_t = st.count_valid(source_map)
    init_physical = dict(st.DEFAULT_REAL_INIT_PHYSICAL)
    reference_advec_lon_abs = float(args.real_reference_advec_lon_abs)
    params_list = st.make_params_list(init_physical, dtype=st.DTYPE, device=device, nugget_mode='zero')

    spec = dict(spec)
    model = RealDataCorridorWidth4x4Lag432NoNuggetGeneralizedCauchyFit(
        gc_alpha=float(spec['gc_alpha']),
        gc_beta=float(spec['gc_beta']),
        input_map=source_map,
        grid_coords=grid_coords_np,
        lag1_lon_offset=reference_advec_lon_abs,
        daily_stride=int(args.daily_stride),
        target_chunk_size=int(args.target_chunk_size),
        min_target_points=int(args.min_target_points),
    )

    t0 = time.time()
    model.precompute_conditioning_sets()
    if strategy_name == 'heads200_points_x8_plus_432':
        install_head_augmented_pseudolikelihood(model, heads_per_time=HEADS_PER_TIME)
    elif strategy_name != 'standard_432':
        raise ValueError(strategy_name)
    precompute_s = time.time() - t0

    base_likelihood = model.vecchia_batched_likelihood
    model._loss_eval_calls = 0

    def _counted_likelihood(params):
        model._loss_eval_calls += 1
        return base_likelihood(params)

    model.vecchia_batched_likelihood = _counted_likelihood

    optimizer = model.set_optimizer(
        params_list,
        lr=float(args.lbfgs_lr),
        max_iter=int(args.lbfgs_eval),
        max_eval=int(args.lbfgs_eval),
        history_size=int(args.lbfgs_history),
    )
    t1 = time.time()
    if args.suppress_fit_prints:
        with contextlib.redirect_stdout(io.StringIO()):
            out, steps_ran = model.fit_vecc_lbfgs(
                params_list, optimizer, max_steps=int(args.lbfgs_steps), grad_tol=float(args.grad_tol)
            )
    else:
        out, steps_ran = model.fit_vecc_lbfgs(
            params_list, optimizer, max_steps=int(args.lbfgs_steps), grad_tol=float(args.grad_tol)
        )
    fit_s = time.time() - t1

    params_tensor = torch.stack([p.reshape(()) for p in params_list]).detach()
    beta = model.get_gls_beta(params_tensor).detach()
    lat_mean = float(model.lat_mean_val)
    est = st.backmap_params(out, nugget_mode='zero')
    cluster_summary = model.cluster_summary()
    loss = float(out[-1])
    est['loss'] = loss

    row = {
        **cluster_summary,
        'fit_id': int(fit_id),
        'strategy': strategy_name,
        'vecchia_strategy': str(cluster_summary.get('strategy', '')),
        'day_idx': int(asset['day_idx']),
        'day': str(asset['day']),
        'model_variant': str(spec['model_variant']),
        'model_label': str(spec['label']),
        'gc_alpha': float(spec['gc_alpha']),
        'gc_beta': float(spec['gc_beta']),
        'nugget_mode': 'zero',
        'n_grid_full': int(grid_coords_np.shape[0]),
        'n_time_slots': int(len(source_map)),
        'n_rows_total': int(n_total),
        'n_valid_o3': int(n_valid),
        'valid_by_t': json.dumps(valid_by_t, separators=(',', ':')),
        'loss': loss,
        'loss_label': f'{loss:.5f}',
        'steps_raw': int(steps_ran),
        'loss_eval_calls': int(getattr(model, '_loss_eval_calls', 0)),
        'heads_accumulate_calls': int(getattr(model, '_heads_accumulate_calls', 0)),
        'heads_accumulate_s': float(getattr(model, '_heads_accumulate_s', 0.0)),
        'precompute_s': float(precompute_s),
        'fit_s': float(fit_s),
        'total_s': float(precompute_s + fit_s),
        'gls_lat_mean': float(lat_mean),
        **{f'est_{k}': float(est[k]) for k in st.P_LABELS},
        **{f'beta_{i}': float(beta.reshape(-1)[i].detach().cpu().item()) for i in range(beta.numel())},
    }
    del model, params_list, optimizer, source_map
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return row, beta, lat_mean

In [6]:
spec = st.variant_spec(MODEL_VARIANT)
strategies = ['standard_432', 'heads200_points_x8_plus_432']
fit_rows = []
profile_rows = []

for fit_id, strategy_name in enumerate(strategies, start=1):
    print('\n' + '-' * 90)
    print(f"strategy={strategy_name} day={asset['day']} model={MODEL_VARIANT} ({spec['label']})")
    print('-' * 90)
    t0 = time.time()
    row, beta, lat_mean = fit_asset_strategy(asset, spec, strategy_name=strategy_name, fit_id=fit_id)
    print(f"loss={row['loss']:.5f}, total_s={row['total_s']:.1f}, sigmasq={row['est_sigmasq']:.4g}, range_lat={row['est_range_lat']:.4g}, range_lon={row['est_range_lon']:.4g}, range_time={row['est_range_time']:.4g}")
    fit_rows.append(row)
    est = {k: float(row[f'est_{k}']) for k in st.P_LABELS}
    rows, stats = st.compute_spectral_profile(
        asset=asset,
        est=est,
        beta=beta,
        lat_mean=lat_mean,
        spec=spec,
        fit_id=int(fit_id),
        device=device,
        args=args,
    )
    row.update(stats)
    for r in rows:
        r['strategy'] = strategy_name
        r['loss'] = row['loss']
        r['loss_label'] = row['loss_label']
    profile_rows.extend(rows)
    print(f"spectral rows={len(rows)}, elapsed={time.time() - t0:.1f}s")

fit_df = pd.DataFrame(fit_rows)
profile_df = pd.DataFrame(profile_rows)
fit_path = OUT_DIR / 'heads_vecchia_lag432_one_day_fit_summary.csv'
profile_path = OUT_DIR / 'heads_vecchia_lag432_one_day_profile_rows.csv'
fit_df.to_csv(fit_path, index=False)
profile_df.to_csv(profile_path, index=False)
print('Saved:', fit_path)
print('Saved:', profile_path)
fit_df[[
    'strategy', 'loss_label', 'steps_raw', 'loss_eval_calls', 'heads_accumulate_calls', 'heads_accumulate_s',
    'precompute_s', 'fit_s', 'total_s',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'n_heads_exact', 'n_target_points', 'n_pseudo_observations'
]]


------------------------------------------------------------------------------------------
strategy=standard_432 day=2024-07-01 model=gc_a075_b1 (GC a=0.75 b=1 nugget0)
------------------------------------------------------------------------------------------
Pre-computing StrategyClusterVecchia (smooth=0.5, strategy=offset_corridor_tapered, block=(4, 4), origin=0/0, lag_blocks=4/3/2, basis=corridor, force_center=0, offsets=0.1260/0.2520, corridors=(0.063, 0.189)/(0.0, 0.252), anchor_mode=width)... Done. clusters=1160, max_points/block=16, target_blocks=8896, target_points=131428, batches=[A:m64:b4x1, A:m64:b8x52, A:m64:b10x1, A:m64:b12x14, A:m64:b16x1092, AB:m112:b1x16, AB:m112:b2x12, AB:m112:b3x8, AB:m112:b4x10, AB:m112:b5x5, ... (37 batches)]
loss=1.19013, total_s=149.8, sigmasq=23.38, range_lat=0.8153, range_lon=0.9527, range_time=4.833
spectral rows=292, elapsed=152.1s

------------------------------------------------------------------------------------------
strategy=heads200_po

,strategy,loss_label,steps_raw,loss_eval_calls,heads_accumulate_calls,heads_accumulate_s,precompute_s,fit_s,total_s,est_sigmasq,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,n_heads_exact,n_target_points,n_pseudo_observations
0,standard_432,1.19013,1,27,0,0.000000,0.852379,148.995372,149.847751,23.377271,0.815299,0.952714,4.833005,-0.012629,-0.194194,NaN,131428,NaN
1,heads200_points_x8_plus_432,1.21514,0,20,21,0.325565,0.911012,113.799394,114.710406,22.463440,0.769189,0.898690,4.534168,-0.012965,-0.194537,1465.0,131428,132893.0


In [7]:
# ST spectral diagnostics from real_july_st_corridor_spectral_profile_v2_061426.py style.
# This uses the profile rows created immediately after fitting.
st_norm = profile_df[profile_df['direction'].eq('norm')].replace([np.inf, -np.inf], np.nan).copy()
if st_norm.empty:
    raise RuntimeError('No norm-frequency profile rows were produced.')

strategy_order = [s for s in strategies if s in set(st_norm['strategy'].astype(str))]
colors = {
    'standard_432': '#1f77b4',
    'heads200_points_x8_plus_432': '#d62728',
}

# Panel A/B: observed marginal spectrum I(k) against profiled diagonal E[I](k).
fig, axes = plt.subplots(1, len(strategy_order), figsize=(7.2 * len(strategy_order), 4.8), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
positive_vals = pd.concat([
    pd.to_numeric(st_norm['data_spectrum_mean'], errors='coerce'),
    pd.to_numeric(st_norm['expected_spectrum_profile_mean'], errors='coerce'),
], ignore_index=True).replace([np.inf, -np.inf], np.nan).dropna()
positive_vals = positive_vals[positive_vals > 0]
ylim = (1e-2, 1e6) if positive_vals.empty else (
    10 ** np.floor(np.log10(float(positive_vals.min()))),
    10 ** np.ceil(np.log10(float(positive_vals.max()))),
)
for ax, strategy_name in zip(axes, strategy_order):
    sub = st_norm[st_norm['strategy'].astype(str).eq(strategy_name)].sort_values('k_mid')
    color = colors.get(strategy_name, None)
    ax.plot(sub['k_mid'], sub['data_spectrum_mean'], color='0.35', lw=1.6, label='I(k) data')
    ax.plot(sub['k_mid'], sub['expected_spectrum_profile_mean'], color=color, lw=2.0, label='profiled E[I](k)')
    ax.set_yscale('log')
    ax.set_ylim(*ylim)
    ax.set_xlabel('norm frequency')
    ax.set_ylabel('marginal spectrum')
    loss_label = sub['loss_label'].iloc[0] if 'loss_label' in sub and len(sub) else ''
    ax.set_title(f'{strategy_name}\nloss={loss_label}')
    ax.grid(alpha=0.25, which='both')
    ax.legend(fontsize=8)
fig.suptitle(f'July {YEAR}: ST marginal I(k) vs profiled E[I](k), {MODEL_VARIANT}')
fig.tight_layout(rect=[0, 0, 1, 0.90])
ie_plot_path = OUT_DIR / 'heads_vecchia_lag432_one_day_norm_I_vs_EI_profiled.png'
fig.savefig(ie_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved I vs EI plot:', ie_plot_path)

# Panel C: same ST script's whitened 8x8 I/E[I] target-1 diagnostic.
fig, ax = plt.subplots(figsize=(7.4, 4.8))
for strategy_name in strategy_order:
    sub = st_norm[st_norm['strategy'].astype(str).eq(strategy_name)].sort_values('k_mid')
    color = colors.get(strategy_name, None)
    x = pd.to_numeric(sub['k_mid'], errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(sub['whitened_ratio_mean'], errors='coerce').to_numpy(dtype=float)
    lo = pd.to_numeric(sub['whitened_ratio_p10'], errors='coerce').to_numpy(dtype=float) if 'whitened_ratio_p10' in sub else np.full_like(y, np.nan)
    hi = pd.to_numeric(sub['whitened_ratio_p90'], errors='coerce').to_numpy(dtype=float) if 'whitened_ratio_p90' in sub else np.full_like(y, np.nan)
    ok_band = np.isfinite(x) & np.isfinite(lo) & np.isfinite(hi) & (lo > 0) & (hi > 0)
    if ok_band.any():
        ax.fill_between(x[ok_band], lo[ok_band], hi[ok_band], color=color, alpha=0.12, linewidth=0)
    label_loss = sub['loss_label'].iloc[0] if 'loss_label' in sub and len(sub) else ''
    ax.plot(x, y, color=color, lw=2.0, label=f'{strategy_name} loss={label_loss}')
ax.axhline(1.0, color='0.25', ls='--', lw=1.0)
ax.set_yscale('log')
ax.set_xlabel('norm frequency')
ax.set_ylabel('8x8 whitened I / E[I] (target = 1)')
ax.set_title(f'July {YEAR}: ST 8x8-whitened spectral ratio, {MODEL_VARIANT}')
ax.grid(alpha=0.25, which='both')
ax.legend(fontsize=8)
fig.tight_layout()
white_plot_path = OUT_DIR / 'heads_vecchia_lag432_one_day_norm_whitened_8x8_ratio.png'
fig.savefig(white_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved whitened ratio plot:', white_plot_path)

# Compact ratio tables following the ST script's band-summary idea.
st_ratio_pivot = (
    st_norm
    .pivot_table(index=['bin_idx', 'k_mid'], columns='strategy', values='whitened_ratio_mean', aggfunc='first')
    .reset_index()
    .sort_values(['bin_idx', 'k_mid'])
)
st_ratio_pivot_path = OUT_DIR / 'heads_vecchia_lag432_one_day_norm_whitened_ratio_pivot.csv'
st_ratio_pivot.to_csv(st_ratio_pivot_path, index=False)
print('Saved whitened ratio pivot:', st_ratio_pivot_path)
display(st_ratio_pivot.head(12).round(5))


Saved I vs EI plot: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626/heads_vecchia_lag432_one_day_norm_I_vs_EI_profiled.png
Saved whitened ratio plot: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626/heads_vecchia_lag432_one_day_norm_whitened_8x8_ratio.png
Saved whitened ratio pivot: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626/heads_vecchia_lag432_one_day_norm_whitened_ratio_pivot.csv


/var/folders/9p/53hd4c7d2fl193h4jwp194wc0000gn/T/ipykernel_35146/2473979382.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/9p/53hd4c7d2fl193h4jwp194wc0000gn/T/ipykernel_35146/2473979382.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


strategy,bin_idx,k_mid,heads200_points_x8_plus_432,standard_432
0,0,0.08645,1.18499,1.26090
1,1,0.25935,1.19329,1.15982
2,2,0.43226,0.88651,0.87006
3,3,0.60516,0.84018,0.83084
4,4,0.77806,0.69494,0.68709
5,5,0.95097,0.79277,0.78648
6,6,1.12387,0.92066,0.91402
7,7,1.29677,0.84604,0.83693
8,8,1.46967,0.96512,0.95680
9,9,1.64258,0.90999,0.90455


In [8]:
# x8 sparse per-time eigen diagnostics, adapted from the pure-space eigen diagnostic script.
EIG_SPARSE_STRIDE = 8
EIG_RIDGE = 1e-8
EIG_RTOL = 1e-10
EIG_ATOL = 1e-12
BROWN_BRIDGE_Q95 = 1.3581015157406195


def beta_from_row(row):
    beta_cols = sorted([c for c in row.index if str(c).startswith('beta_')], key=lambda x: int(str(x).split('_')[1]))
    return torch.tensor([float(row[c]) for c in beta_cols], dtype=st.DTYPE, device=device).reshape(-1, 1)


def sparse_x8_valid_rows(rows: torch.Tensor, stride: int = EIG_SPARSE_STRIDE):
    arr = rows.detach().cpu().numpy()
    valid = np.isfinite(arr[:, 0]) & np.isfinite(arr[:, 1]) & np.isfinite(arr[:, 2])
    if not np.any(valid):
        return np.empty((0, 2)), np.empty((0,))
    lat_key = np.round(arr[:, 0], 10)
    lon_key = np.round(arr[:, 1], 10)
    lat_vals = np.sort(np.unique(lat_key[np.isfinite(lat_key)]))
    lon_vals = np.sort(np.unique(lon_key[np.isfinite(lon_key)]))
    lat_to_row = {float(v): i for i, v in enumerate(lat_vals)}
    lon_to_col = {float(v): i for i, v in enumerate(lon_vals)}
    rr = np.asarray([lat_to_row.get(float(v), -1) for v in lat_key], dtype=np.int64)
    cc = np.asarray([lon_to_col.get(float(v), -1) for v in lon_key], dtype=np.int64)
    keep = valid & (rr >= 0) & (cc >= 0) & (rr % int(stride) == 0) & (cc % int(stride) == 0)
    sub = arr[keep]
    coords = sub[:, 0:2].astype(np.float64)
    z = sub[:, 2].astype(np.float64)
    return coords, z


def eigen_design_matrix(coords: np.ndarray) -> np.ndarray:
    lat = coords[:, 0].astype(np.float64)
    lat_c = lat - float(np.nanmean(lat))
    return np.column_stack([np.ones(len(coords)), lat_c])


def st_spatial_marginal_covariance(coords: np.ndarray, est: dict, spec: dict, jitter: float = EIG_RIDGE) -> np.ndarray:
    coords_t = torch.as_tensor(coords, dtype=st.DTYPE, device=device)
    diff = coords_t.unsqueeze(1) - coords_t.unsqueeze(0)
    range_lat = max(float(est['range_lat']), 1e-12)
    range_lon = max(float(est['range_lon']), 1e-12)
    scaled = torch.sqrt((diff[..., 0] / range_lat).pow(2) + (diff[..., 1] / range_lon).pow(2) + 1e-12)
    alpha = float(spec['gc_alpha'])
    beta = float(spec['gc_beta'])
    corr = torch.pow(1.0 + torch.pow(scaled.clamp_min(1e-10), alpha), -beta / alpha)
    cov = float(est['sigmasq']) * corr
    cov.diagonal().add_(float(jitter))
    return cov.detach().cpu().numpy()


def eigen_diagnostic_sparse_time(z: np.ndarray, coords: np.ndarray, est: dict, spec: dict):
    n = int(len(z))
    if n < 8:
        raise ValueError(f'Not enough sparse points for eigen diagnostic: n={n}')
    X = eigen_design_matrix(coords)
    p_rank = int(np.linalg.matrix_rank(X))
    if p_rank >= n:
        raise ValueError(f'Mean rank {p_rank} is not smaller than n={n}')
    q, _ = np.linalg.qr(X, mode='reduced')
    rz = z - q @ (q.T @ z)
    sigma = st_spatial_marginal_covariance(coords, est, spec)
    proj = np.eye(n) - q @ q.T
    k_mat = proj @ sigma @ proj
    k_mat = 0.5 * (k_mat + k_mat.T)
    evals, evecs = np.linalg.eigh(k_mat)
    max_eval = max(float(np.nanmax(evals)), 1e-12)
    threshold = max(float(EIG_ATOL), float(EIG_RTOL) * max_eval)
    keep = evals > threshold
    evals = evals[keep]
    evecs = evecs[:, keep]
    if evals.size == 0:
        raise ValueError('No positive projected covariance eigenvalues survived threshold')
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]
    scores = (evecs.T @ rz) / np.sqrt(evals)
    y2 = scores ** 2
    csum = np.cumsum(y2)
    m = int(len(y2))
    index = np.arange(1, m + 1, dtype=np.int64)
    width = float(BROWN_BRIDGE_Q95) * math.sqrt(2.0 * m)
    bridge_d = float(np.max(np.abs(csum - index)) / math.sqrt(2.0 * m))
    curve = pd.DataFrame({
        'index': index,
        'frac_index': index.astype(float) / float(m),
        'eigenvalue': evals,
        'y2': y2,
        'cumsum_y2': csum,
        'scaled_cumsum': csum / float(m),
        'expected': index.astype(float),
        'band_lower': index.astype(float) - width,
        'band_upper': index.astype(float) + width,
    })
    summary = {
        'n_obs': n,
        'mean_rank': p_rank,
        'n_eigen': m,
        'eigen_threshold': threshold,
        'min_kept_eigen': float(np.nanmin(evals)),
        'max_kept_eigen': float(np.nanmax(evals)),
        'sum_y2': float(csum[-1]),
        'mean_y2': float(np.nanmean(y2)),
        'max_abs_bridge_scaled': bridge_d,
        'brown_bridge_width': width,
    }
    return curve, summary


def plot_eigen_curve_notebook(curve: pd.DataFrame, title: str, out_path: Path):
    m = int(curve['index'].max())
    y_max = 1.03 * max(float(m), float(np.nanmax(curve['cumsum_y2'].to_numpy(dtype=float))))
    fig, ax = plt.subplots(figsize=(5.4, 5.0))
    ax.scatter(curve['index'], curve['cumsum_y2'], s=11, facecolors='none', edgecolors='black', linewidths=0.8, alpha=0.85)
    ax.plot([0, m], [0, m], color='0.45', linewidth=1.5)
    ax.plot(curve['index'], curve['band_lower'], color='0.60', linestyle=(0, (4, 4)), linewidth=1.1)
    ax.plot(curve['index'], curve['band_upper'], color='0.60', linestyle=(0, (4, 4)), linewidth=1.1)
    ax.set_xlim(0, m)
    ax.set_ylim(0, y_max)
    ax.set_xlabel('index')
    ax.set_ylabel('cumulative sum')
    ax.set_title(title)
    ax.grid(alpha=0.20)
    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


def resample_eigen_curve(curve: pd.DataFrame, n_grid: int = 200) -> pd.DataFrame:
    m = float(curve['index'].max())
    grid = np.linspace(1.0 / n_grid, 1.0, n_grid)
    vals = np.interp(grid, curve['frac_index'].to_numpy(float), curve['scaled_cumsum'].to_numpy(float))
    return pd.DataFrame({'frac_index': grid, 'scaled_cumsum': vals, 'n_eigen': m})


In [9]:
# Run x8 sparse eigen diagnostics for each time slot and strategy. No tiles.
eig_dir = OUT_DIR / 'x8_sparse_time_eigen'
eig_dir.mkdir(parents=True, exist_ok=True)

eig_summary_rows = []
eig_avg_rows = []
for row in fit_df.itertuples(index=False):
    row_s = pd.Series(row._asdict())
    strategy_name = str(row_s['strategy'])
    beta_t = beta_from_row(row_s)
    lat_mean = float(row_s['gls_lat_mean'])
    est = {k: float(row_s[f'est_{k}']) for k in st.P_LABELS}
    est['sigmasq'] = est['sigmasq']
    est['gc_alpha'] = float(row_s['gc_alpha'])
    est['gc_beta'] = float(row_s['gc_beta'])
    slices = st.residual_time_slices(asset, beta=beta_t, lat_mean=lat_mean, device=device)
    for time_idx, rows_t in enumerate(slices):
        coords, z = sparse_x8_valid_rows(rows_t, stride=EIG_SPARSE_STRIDE)
        try:
            curve, summary = eigen_diagnostic_sparse_time(z=z, coords=coords, est=est, spec=spec)
        except Exception as exc:
            eig_summary_rows.append({
                'strategy': strategy_name,
                'time_idx': int(time_idx),
                'status': 'error',
                'error': f'{type(exc).__name__}: {exc}',
                'n_obs': int(len(z)),
            })
            continue
        curve_path = eig_dir / f'{strategy_name}_time{time_idx:02d}_x8_eigen_curve.csv'
        curve.assign(strategy=strategy_name, time_idx=int(time_idx)).to_csv(curve_path, index=False)
        plot_path = eig_dir / f'{strategy_name}_time{time_idx:02d}_x8_eigen_curve.png'
        plot_eigen_curve_notebook(
            curve,
            f'{strategy_name}, time {time_idx}, x8 sparse\nD={summary["max_abs_bridge_scaled"]:.2f}, n={summary["n_obs"]}, m={summary["n_eigen"]}',
            plot_path,
        )
        eig_summary_rows.append({
            'strategy': strategy_name,
            'time_idx': int(time_idx),
            'status': 'ok',
            **summary,
            'curve_csv': str(curve_path),
            'curve_png': str(plot_path),
        })
        eig_avg_rows.append(resample_eigen_curve(curve).assign(strategy=strategy_name, time_idx=int(time_idx)))
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

eig_summary_df = pd.DataFrame(eig_summary_rows)
eig_summary_path = eig_dir / 'x8_sparse_time_eigen_summary.csv'
eig_summary_df.to_csv(eig_summary_path, index=False)
print('Saved x8 sparse time eigen summary:', eig_summary_path)
display(eig_summary_df)

if eig_avg_rows:
    eig_avg_df = pd.concat(eig_avg_rows, ignore_index=True)
    eig_mean = (
        eig_avg_df
        .groupby(['strategy', 'frac_index'], as_index=False)
        .agg(
            scaled_cumsum_mean=('scaled_cumsum', 'mean'),
            scaled_cumsum_std=('scaled_cumsum', 'std'),
            n_curves=('scaled_cumsum', 'count'),
            n_eigen_mean=('n_eigen', 'mean'),
        )
    )
    eig_mean_path = eig_dir / 'x8_sparse_time_eigen_average_curves.csv'
    eig_mean.to_csv(eig_mean_path, index=False)
    print('Saved x8 sparse time eigen average curves:', eig_mean_path)

    fig, ax = plt.subplots(figsize=(6.4, 5.2))
    for strategy_name, sub in eig_mean.groupby('strategy', sort=False):
        sub = sub.sort_values('frac_index')
        color = colors.get(strategy_name, None)
        x = sub['frac_index'].to_numpy(float)
        y = sub['scaled_cumsum_mean'].to_numpy(float)
        sd = sub['scaled_cumsum_std'].fillna(0.0).to_numpy(float)
        ax.plot(x, y, color=color, linewidth=2.0, label=strategy_name)
        ax.fill_between(x, y - sd, y + sd, color=color, alpha=0.12, linewidth=0)
    ax.plot([0, 1], [0, 1], color='0.50', linewidth=1.1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, max(1.05, float(np.nanmax(eig_mean['scaled_cumsum_mean'].to_numpy(float))) * 1.05))
    ax.set_xlabel('eigenvalue rank fraction')
    ax.set_ylabel('cumulative sum / m')
    ax.set_title(f'July {YEAR}: x8 sparse per-time eigen diagnostic, spatial marginal ST covariance')
    ax.grid(alpha=0.20)
    ax.legend(fontsize=8)
    fig.tight_layout()
    eig_avg_plot_path = eig_dir / 'x8_sparse_time_eigen_average_by_strategy.png'
    fig.savefig(eig_avg_plot_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved x8 sparse time eigen average plot:', eig_avg_plot_path)


Saved x8 sparse time eigen summary: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626/x8_sparse_time_eigen/x8_sparse_time_eigen_summary.csv


,strategy,time_idx,status,n_obs,mean_rank,n_eigen,eigen_threshold,min_kept_eigen,max_kept_eigen,sum_y2,mean_y2,max_abs_bridge_scaled,brown_bridge_width,curve_csv,curve_png
0,standard_432,0,ok,300,2,298,4.104699e-08,7.665684,410.469901,252.009464,0.845669,2.164974,33.155483,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
1,standard_432,1,ok,266,2,264,3.514646e-08,7.674013,351.464632,234.143531,0.886907,1.299336,31.206797,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
2,standard_432,2,ok,242,2,240,3.272897e-08,7.707163,327.289704,232.626247,0.969276,0.938836,29.754513,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
3,standard_432,3,ok,241,2,239,3.383862e-08,7.696109,338.386205,200.471781,0.838794,1.981567,29.692460,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
4,standard_432,4,ok,250,2,248,3.520573e-08,7.700312,352.057328,220.731595,0.890047,1.246322,30.246357,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
5,standard_432,5,ok,284,2,282,3.879083e-08,7.674912,387.908346,276.680992,0.981138,0.806436,32.253124,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
6,standard_432,6,ok,298,2,296,4.078170e-08,7.665769,407.816966,283.496097,0.957757,1.052337,33.044036,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
7,standard_432,7,ok,298,2,296,4.078170e-08,7.665769,407.816966,252.009556,0.851384,1.913234,33.044036,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
8,heads200_points_x8_plus_432,0,ok,300,2,298,3.845914e-08,7.608164,384.591391,256.076390,0.859317,2.027362,33.155483,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...
9,heads200_points_x8_plus_432,1,ok,266,2,264,3.294102e-08,7.616289,329.410234,238.169989,0.902159,1.156774,31.206797,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...,/Users/joonwonlee/Documents/GEMS_TCO-1/Exercis...


Saved x8 sparse time eigen average curves: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626/x8_sparse_time_eigen/x8_sparse_time_eigen_average_curves.csv
Saved x8 sparse time eigen average plot: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/space_time/spectrum_diagnostics/outputs/real_july2024_st_heads_vecchia_lag432_one_day_gc_a075_b1_nugget0_061626/x8_sparse_time_eigen/x8_sparse_time_eigen_average_by_strategy.png


/var/folders/9p/53hd4c7d2fl193h4jwp194wc0000gn/T/ipykernel_35146/2946056027.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
summary_cols = [
    'strategy', 'loss_label', 'steps_raw', 'loss_eval_calls', 'heads_accumulate_calls', 'heads_accumulate_s',
    'precompute_s', 'fit_s', 'total_s',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time',
    'est_advec_lat', 'est_advec_lon', 'n_heads_exact', 'n_target_points', 'n_pseudo_observations',
    'spectral_global_scale', 'spectral_data_over_expected_mean', 'spectral_ratio_var_after_profile',
]
fit_df[summary_cols].style.format({
    'heads_accumulate_s': '{:.1f}', 'precompute_s': '{:.1f}', 'fit_s': '{:.1f}', 'total_s': '{:.1f}',
    'est_sigmasq': '{:.4g}', 'est_range_lat': '{:.4g}', 'est_range_lon': '{:.4g}', 'est_range_time': '{:.4g}',
    'est_advec_lat': '{:.5f}', 'est_advec_lon': '{:.5f}',
    'spectral_global_scale': '{:.4g}', 'spectral_data_over_expected_mean': '{:.4g}', 'spectral_ratio_var_after_profile': '{:.4g}',
})


,strategy,loss_label,steps_raw,loss_eval_calls,heads_accumulate_calls,heads_accumulate_s,precompute_s,fit_s,total_s,est_sigmasq,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,n_heads_exact,n_target_points,n_pseudo_observations,spectral_global_scale,spectral_data_over_expected_mean,spectral_ratio_var_after_profile
0,standard_432,1.19013,1,27,0,0.0,0.9,149.0,149.8,23.38,0.8153,0.9527,4.833,-0.01263,-0.19419,nan,131428,nan,0.9322,0.9207,0.1934
1,heads200_points_x8_plus_432,1.21514,0,20,21,0.3,0.9,113.8,114.7,22.46,0.7692,0.8987,4.534,-0.01296,-0.19454,1465.000000,131428,132893.000000,0.9368,0.9255,0.1943
